# Diagnose der Koreferenzauflösung

## Konfiguration

In [1]:
from pathlib import Path
import json

AUSGABE   = (Path("..") / "05_ergebnisse" / "koref").resolve()
ZWISCHEN  = AUSGABE / "aufgeloest"
FEHLER    = AUSGABE / "fehlerliste_koref.csv"
GOLD_PFAD = (Path("..") / "03_annotation_laufend" / "pronomen"
             / "A4_pronomen_final.csv").resolve()

protokolldateien = sorted(ZWISCHEN.glob("*_protokoll.json"))
print(f"Protokolle: {len(protokolldateien)}")
print(f"Fehlerliste: {'vorhanden' if FEHLER.exists() else 'FEHLT'}")
assert protokolldateien, f"keine Protokolle unter {ZWISCHEN}"
assert FEHLER.exists(), "Fehlerliste fehlt; zuerst koref_eval laufen lassen"

Protokolle: 20
Fehlerliste: vorhanden


## Schritt 1 — Protokolle einlesen

In [2]:
import re, pandas as pd
from collections import Counter


def schluessel(name):
    roh = name.replace("-mod", "").replace("_protokoll", "").lower()
    return re.sub(r"[^0-9a-z\u00e4\u00f6\u00fc\u00df]", "", roh)


zeilen = []
for pfad in protokolldateien:
    p = json.loads(pfad.read_text(encoding="utf-8"))
    fenster = p.get("fenster", [])
    cluster_gesamt = sum(len(f.get("cluster", [])) for f in fenster)
    ohne_figur = sum(f.get("cluster_ohne_figur", 0) for f in fenster)
    mit_rep = sum(1 for f in fenster for c in f.get("cluster", [])
                  if c.get("repraesentant"))
    ersetzungen = Counter()
    for f in fenster:
        ersetzungen.update(f.get("ersetzungen", {}) or {})
    zeilen.append({
        "datei": schluessel(pfad.stem),
        "fenster": len(fenster),
        "figuren": sum(f.get("anzahl_figuren", 0) for f in fenster),
        "cluster": cluster_gesamt,
        "cluster_mit_repraesentant": mit_rep,
        "cluster_ohne_figur": ohne_figur,
        "ersetzungen_gesamt": int(sum(ersetzungen.values())),
        "possessiva_offen": len([q for q in p.get("possessiva", [])
                                 if not q.get("im_kontext")]),
    })

prot = pd.DataFrame(zeilen).set_index("datei")
prot["anteil_mit_repraesentant"] = (
    prot["cluster_mit_repraesentant"] / prot["cluster"].clip(lower=1))
print(prot[["fenster", "figuren", "cluster", "cluster_mit_repraesentant",
            "ersetzungen_gesamt"]].to_string())

                                                              fenster  figuren  cluster  cluster_mit_repraesentant  ersetzungen_gesamt
datei                                                                                                                                 
0alaeddinunddiewunderlampe                                         72     1110      459                        459               10603
0dieabenteuersindbadsdesseefahrers                                 56      836      363                        363                8256
0diegeschichtevonalidemkairenserunddemspukendenhauseinbagdad       15      330      167                        167                3560
0dschudarundseinebrüder                                            33      641      350                        350                8063
0khalifahderfischervonbagdad                                       16      330      157                        157                4267
1dereisenhans                                          

## Schritt 2 — Fehlerklassen je Erzählung

In [3]:
gold = pd.read_csv(GOLD_PFAD)
fehler = pd.read_csv(FEHLER)

je_erz = []
for erz, teil in gold.groupby("erzaehlung"):
    f = fehler[fehler["erzaehlung"] == erz]
    je_erz.append({
        "datei": schluessel(erz),
        "erzaehlung": erz,
        "einheiten": len(teil),
        "nicht_ersetzt": int((f["klasse"] == "nicht_ersetzt").sum()),
        "falsche_figur": int((f["klasse"] == "falsche_figur").sum()),
    })
erg = pd.DataFrame(je_erz).set_index("datei")

tab = erg.join(prot, how="inner")
tab["quote_nicht_ersetzt"] = tab["nicht_ersetzt"] / tab["einheiten"]
print(f"Verknuepft: {len(tab)} von {len(erg)} Erzaehlungen")
fehlend = set(erg.index) - set(prot.index)
if fehlend:
    print("ohne Protokoll:", ", ".join(sorted(fehlend)))

Verknuepft: 20 von 20 Erzaehlungen


## Schritt 3 — Die Vermutung prüfen

In [4]:
import numpy as np

spalten = ["einheiten", "nicht_ersetzt", "falsche_figur", "figuren",
           "cluster", "cluster_mit_repraesentant", "anteil_mit_repraesentant",
           "ersetzungen_gesamt"]
zeigen = tab.sort_values("quote_nicht_ersetzt", ascending=False)
print(f"{'Erzaehlung':<40}{'Einh':>5}{'n.ers':>6}{'falsch':>7}"
      f"{'Fig':>5}{'Clust':>6}{'m.Rep':>6}{'Anteil':>8}{'Ersetz':>7}")
for _, r in zeigen.iterrows():
    print(f"{r['erzaehlung'][:40]:<40}{int(r['einheiten']):>5}"
          f"{int(r['nicht_ersetzt']):>6}{int(r['falsche_figur']):>7}"
          f"{int(r['figuren']):>5}{int(r['cluster']):>6}"
          f"{int(r['cluster_mit_repraesentant']):>6}"
          f"{r['anteil_mit_repraesentant']:>8.2f}"
          f"{int(r['ersetzungen_gesamt']):>7}")


def rangkorrelation(a, b):
    ra = pd.Series(a).rank().to_numpy()
    rb = pd.Series(b).rank().to_numpy()
    if len(ra) < 3:
        return float("nan")
    return float(np.corrcoef(ra, rb)[0, 1])


print()
print("Rangkorrelationen mit dem Anteil unterlassener Ersetzungen:")
for spalte in ("anteil_mit_repraesentant", "figuren", "cluster",
               "ersetzungen_gesamt", "fenster"):
    r = rangkorrelation(tab["quote_nicht_ersetzt"], tab[spalte])
    print(f"  {spalte:<28}{r:>7.3f}")

Erzaehlung                               Einh n.ers falsch  Fig Clust m.Rep  Anteil Ersetz
0_Khalifah der Fischer von Bagdad           9     7      1  330   157   157    1.00   4267
0_Dschudar und seine Brüder                15     7      4  641   350   350    1.00   8063
2_Der Engel                                 9     4      1   33     9     9    1.00    106
2_Die roten Schuhe                         17     7      2   64    22    22    1.00    359
1_Der Räuberbräutigam                      10     3      3   36    22    22    1.00    309
0_Die Abenteuer Sindbads des Seefahrers     7     2      1  836   363   363    1.00   8256
3_Der Wunderkessel                         16     4      0   20    10    10    1.00    132
3_Momotaro                                 13     2      1   69    29    29    1.00    377
1_Märchen von einem, der auszog das Fürc   14     2      5   73    55    55    1.00   1118
2_Das Märchen vom Sandmann                  8     1      0   97    36    36    1.00    469

/home/timc/Workspace/fairytale-personality-profiling/venv_cuda_test/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/timc/Workspace/fairytale-personality-profiling/venv_cuda_test/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


## Schritt 4 — Gegenprobe an den Einzelfällen

In [5]:
def repraesentanten(datei):
    pfad = next((p for p in protokolldateien
                 if schluessel(p.stem) == datei), None)
    if pfad is None:
        return []
    p = json.loads(pfad.read_text(encoding="utf-8"))
    namen = Counter()
    for f in p.get("fenster", []):
        for c in f.get("cluster", []):
            if c.get("repraesentant"):
                namen[c["repraesentant"]] += len(c.get("mentions", []))
    return namen.most_common(8)


viel_offen = tab.nlargest(2, "quote_nicht_ersetzt")
viel_falsch = tab.nlargest(2, "falsche_figur")

for titel, teil in (("Ersetzung unterbleibt haeufig", viel_offen),
                    ("Ersetzung trifft haeufig daneben", viel_falsch)):
    print(f"══ {titel} ══")
    for datei, r in teil.iterrows():
        print(f"  {r['erzaehlung']}")
        print(f"    Einheiten {int(r['einheiten'])} | nicht ersetzt "
              f"{int(r['nicht_ersetzt'])} | falsche Figur "
              f"{int(r['falsche_figur'])}")
        print(f"    Cluster {int(r['cluster'])}, davon mit Repraesentant "
              f"{int(r['cluster_mit_repraesentant'])}")
        namen = repraesentanten(datei)
        print("    Repraesentanten: "
              + (", ".join(f"{n} ({k})" for n, k in namen) or "keine"))
    print()

══ Ersetzung unterbleibt haeufig ══
  0_Khalifah der Fischer von Bagdad
    Einheiten 9 | nicht ersetzt 7 | falsche Figur 1
    Cluster 157, davon mit Repraesentant 157
    Repraesentanten: Khalifah (2045), Fischer (1098), Jude (164), Kalif (138), Kut al-Kulub (107), Herrin (88), Affe (82), Eselin (69)
  0_Dschudar und seine Brüder
    Einheiten 15 | nicht ersetzt 7 | falsche Figur 4
    Cluster 350, davon mit Repraesentant 350
    Repraesentanten: Dschudar (2965), Mutter (775), Brüder (508), Gast (469), König (415), Mohr (410), Bruder (323), Vezier (185)

══ Ersetzung trifft haeufig daneben ══
  3_Das Mädchen mit dem Holznapfe
    Einheiten 12 | nicht ersetzt 0 | falsche Figur 6
    Cluster 19, davon mit Repraesentant 19
    Repraesentanten: Mädchen (128), Mann (43), Mutter (32), Mannes (15), Ehepaar (11), Frau (4), Kindern (3), Eltern (2)
  1_Die Gänsehirtin am Brunnen
    Einheiten 10 | nicht ersetzt 1 | falsche Figur 5
    Cluster 42, davon mit Repraesentant 42
    Repraesentanten:

## Schritt 5 — Welche Referenten fehlen im Angebot

In [ ]:
def norm(s):
    return re.sub(r"[^0-9a-z\u00e4\u00f6\u00fc\u00df]", "", str(s).lower())


angebot = {d: {norm(n) for n, _ in repraesentanten(d)} for d in tab.index}
angebot_voll = {}
for pfad in protokolldateien:
    p = json.loads(pfad.read_text(encoding="utf-8"))
    namen = set()
    for f in p.get("fenster", []):
        for c in f.get("cluster", []):
            if c.get("repraesentant"):
                namen.add(norm(c["repraesentant"]))
    angebot_voll[schluessel(pfad.stem)] = namen

offen = fehler[fehler["klasse"] == "nicht_ersetzt"].copy()
offen["datei"] = offen["erzaehlung"].map(schluessel)
offen["referent_im_angebot"] = [
    norm(r["referent_gold"]) in angebot_voll.get(r["datei"], set())
    for _, r in offen.iterrows()]

n_ja = int(offen["referent_im_angebot"].sum())
print(f"Unterlassene Ersetzungen: {len(offen)}")
print(f"  erwarteter Referent war als Repraesentant vorhanden: {n_ja}")
print(f"  war nicht vorhanden:                                {len(offen)-n_ja}")
print()
print("Ohne Ersetzung, obwohl der Referent verfuegbar war (Auswahl):")
for _, r in offen[offen["referent_im_angebot"]].head(8).iterrows():
    print(f"  {r['erzaehlung'][:36]:<36} {r['pronomen']:<8} "
          f"-> erwartet '{r['referent_gold']}'")
print()
print("Ohne Ersetzung, weil der Referent nicht erkannt wurde (Auswahl):")
for _, r in offen[~offen["referent_im_angebot"]].head(8).iterrows():
    print(f"  {r['erzaehlung'][:36]:<36} {r['pronomen']:<8} "
          f"-> erwartet '{r['referent_gold']}'")

Unterlassene Ersetzungen: 44
  erwarteter Referent war als Repraesentant vorhanden: 20
  war nicht vorhanden:                                24

Ohne Ersetzung, obwohl der Referent verfuegbar war (Auswahl):
  2_Die roten Schuhe                   ihr      -> erwartet 'Karen'
  2_Die roten Schuhe                   sie      -> erwartet 'Karen'
  0_Die Abenteuer Sindbads des Seefahr ihnen    -> erwartet 'Kaufleute'
  0_Die Geschichte von Ali dem Kairens Die      -> erwartet 'Sklaven'
  0_Dschudar und seine Brüder          er       -> erwartet 'Fischer'
  0_Dschudar und seine Brüder          ihn      -> erwartet 'Dschudar'
  0_Dschudar und seine Brüder          er       -> erwartet 'Dschudar'
  0_Khalifah der Fischer von Bagdad    ihm      -> erwartet 'Khalifah'

Ohne Ersetzung, weil der Referent nicht erkannt wurde (Auswahl):
  2_Das Märchen vom Sandmann           es       -> erwartet 'Portrait'
  2_Der Engel                          Sie      -> erwartet 'Kind & Engel'
  2_Der Engel       

### Alaeddins Zeile in Schritt 3 auf Artefakte überprüfen, bzgl. des Vorkommens von 10603.

In [ ]:
import json
p = json.load(open("../05_ergebnisse/koref/aufgeloest/"
                   "0_Alaeddin und die Wunderlampe-mod_protokoll.json",
                   encoding="utf-8"))
f = p["fenster"][0]
print(f["ersetzungen"])
print("Chunk-Zeichen:", f["chunk_zeichen"], "| Kontext:", f["kontext_zeichen"])

{'fall3_artikelwort': 2, 'pronomen_behalten': 11, 'einzeltoken_ersetzt': 44, 'possessiv_protokolliert': 30, 'pronomen_ersetzt': 53, 'keine_regel': 10, 'fall2_uebersprungen': 31}
Chunk-Zeichen: 3883 | Kontext: 0
